In [1]:
import sys, os, glob
import numpy as np
import pandas as pd

sys.path.insert(0, '/g/data/gv90/xl1657/phd/M2_workspace/notebooks')

# Force reload of utils in case it was cached
if 'utils' in sys.modules:
    del sys.modules['utils']

from utils import read_atl10_v7, read_cs2_l2e, collocate_cs2_is2, compute_snow_thickness

# === FORMULA VERIFICATION ===
# Test with known values
rho_test = 330.0
rho_gcm3 = rho_test / 1000.0
eta_s_test = (1.0 + 0.51 * rho_gcm3) ** 1.5

print("=" * 50)
print("FORMULA VERIFICATION")
print("=" * 50)
print(f"rho_s = {rho_test} kg/m3 = {rho_gcm3} g/cm3")
print(f"eta_s = (1 + 0.51 * {rho_gcm3})^1.5 = {eta_s_test:.4f}")
print(f"Expected: 1.2634")
print(f"Match: {abs(eta_s_test - 1.2634) < 0.001}")
print()

# Test the actual function
test_is2 = np.array([0.30])
test_cs2 = np.array([0.17])
test_hs, test_unc = compute_snow_thickness(test_is2, test_cs2)
expected_hs = (0.30 - 0.17) / eta_s_test

print(f"Test: IS2_fb=0.30, CS2_rfb=0.17, delta_f=0.13")
print(f"  compute_snow_thickness returns: hs = {test_hs[0]:.4f} m")
print(f"  Manual calculation: 0.13 / {eta_s_test:.4f} = {expected_hs:.4f} m")
print(f"  Match: {abs(test_hs[0] - expected_hs) < 0.0001}")
print()

# CRITICAL CHECK: hs must be LESS than delta_f
print(f"  hs ({test_hs[0]:.4f}) < delta_f (0.13): {test_hs[0] < 0.13}")
if test_hs[0] > 0.13:
    print("  *** FAIL: Snow thickness > freeboard difference! Formula still wrong! ***")
else:
    print("  PASS: Snow thickness correctly less than freeboard difference")

FORMULA VERIFICATION
rho_s = 330.0 kg/m3 = 0.33 g/cm3
eta_s = (1 + 0.51 * 0.33)^1.5 = 1.2628
Expected: 1.2634
Match: True

Test: IS2_fb=0.30, CS2_rfb=0.17, delta_f=0.13
  compute_snow_thickness returns: hs = 0.1029 m
  Manual calculation: 0.13 / 1.2628 = 0.1029 m
  Match: True

  hs (0.1029) < delta_f (0.13): True
  PASS: Snow thickness correctly less than freeboard difference


In [2]:
ATL10_DIR = '/g/data/gv90/xl1657/phd/M2_workspace/data/raw/ATL10'
CS2_DIR   = '/g/data/gv90/xl1657/phd/M2_workspace/data/raw/CS2_L2E'

ym = '202208'

# Find files
a_files = sorted(glob.glob(f'{ATL10_DIR}/**/*{ym}*.h5', recursive=True))
if not a_files:
    a_files = sorted(glob.glob(f'{ATL10_DIR}/*{ym}*.h5'))
c_files = sorted(glob.glob(f'{CS2_DIR}/**/*{ym}*.nc', recursive=True))
if not c_files:
    c_files = sorted(glob.glob(f'{CS2_DIR}/*{ym}*.nc'))

print(f"August 2022 files found:")
print(f"  ATL10: {len(a_files)}")
print(f"  CS2:   {len(c_files)}")

August 2022 files found:
  ATL10: 346
  CS2:   3390


In [3]:
# Extract dates from filenames
def extract_date(fname):
    base = os.path.basename(fname)
    for part in base.split('_'):
        if len(part) >= 8 and part[:8].isdigit():
            return part[:8]
        if 'T' in part and len(part) >= 15:
            return part[:8]
    return None

atl10_by_date = {}
for f in a_files:
    d = extract_date(f)
    if d:
        atl10_by_date.setdefault(d, []).append(f)

cs2_by_date = {}
for f in c_files:
    d = extract_date(f)
    if d:
        cs2_by_date.setdefault(d, []).append(f)

common_dates = sorted(set(atl10_by_date.keys()) & set(cs2_by_date.keys()))
print(f"Common dates in Aug 2022: {len(common_dates)}")
print(f"First 5: {common_dates[:5]}")

# Process just the first common date
test_date = common_dates[0]
print(f"\nProcessing test date: {test_date}")
print(f"  ATL10 files: {len(atl10_by_date[test_date])}")
print(f"  CS2 files:   {len(cs2_by_date[test_date])}")

# Read data
is2_frames = [read_atl10_v7(f) for f in atl10_by_date[test_date]]
is2_frames = [f for f in is2_frames if len(f) > 0]
cs2_frames = [read_cs2_l2e(f) for f in cs2_by_date[test_date]]
cs2_frames = [f for f in cs2_frames if len(f) > 0]

print(f"  Valid IS2 frames: {len(is2_frames)}")
print(f"  Valid CS2 frames: {len(cs2_frames)}")

if is2_frames and cs2_frames:
    df_is2 = pd.concat(is2_frames, ignore_index=True)
    df_cs2 = pd.concat(cs2_frames, ignore_index=True)
    print(f"  IS2 segments: {len(df_is2)}")
    print(f"  CS2 points:   {len(df_cs2)}")

Common dates in Aug 2022: 31
First 5: ['20220801', '20220802', '20220803', '20220804', '20220805']

Processing test date: 20220801
  ATL10 files: 13
  CS2 files:   102
  Beam gt3l not found: 'Unable to synchronously open object (component not found)'
  Valid IS2 frames: 13
  Valid CS2 frames: 23
  IS2 segments: 2954745
  CS2 points:   18986


In [4]:
if is2_frames and cs2_frames:
    matched = collocate_cs2_is2(df_cs2, df_is2, R_m=5000, min_pts=10)
    print(f"Matched CS2-IS2 pairs: {len(matched)}")
    
    if len(matched) > 0:
        # Compute snow thickness with corrected formula
        hs, hs_unc = compute_snow_thickness(
            matched['is2_fb_wm'].values,
            matched['cs2_rfb'].values,
            is2_unc=matched['is2_fb_unc_mean'].values,
            cs2_unc=0.03 * np.ones(len(matched)),
            rho_s=330.0,
            rho_s_unc=70.0,
        )
        
        delta_f = matched['is2_fb_wm'].values - matched['cs2_rfb'].values
        valid = hs > 0
        
        print()
        print("=" * 50)
        print(f"AUGUST 2022 - DATE {test_date} - RESULTS")
        print("=" * 50)
        print(f"Matchups: {len(matched)}")
        print(f"Valid (hs > 0): {valid.sum()} ({100*valid.sum()/len(hs):.1f}%)")
        print()
        print(f"Freeboard difference (delta_f):")
        print(f"  Median: {np.nanmedian(delta_f):.4f} m")
        print(f"  Mean:   {np.nanmean(delta_f):.4f} m")
        print()
        print(f"Snow thickness (CORRECTED):")
        print(f"  Median: {np.nanmedian(hs[valid]):.4f} m")
        print(f"  Mean:   {np.nanmean(hs[valid]):.4f} m")
        print()
        
        # Critical ratio check
        ratio = np.nanmedian(hs[valid]) / np.nanmedian(delta_f[valid])
        print(f"RATIO CHECK:")
        print(f"  hs_median / delta_f_median = {ratio:.4f}")
        print(f"  Expected (1/eta_s = 1/1.2634) = {1/1.2634:.4f}")
        print(f"  PASS: {abs(ratio - 1/1.2634) < 0.02}")
        print()
        
        if hs_unc is not None:
            print(f"Uncertainty:")
            print(f"  Median: {np.nanmedian(hs_unc[valid]):.4f} m")
            print(f"  Mean:   {np.nanmean(hs_unc[valid]):.4f} m")
        print()
        
        # Comparison with old values
        print("COMPARISON WITH OLD (WRONG) PIPELINE:")
        old_eta = 1.0 / np.sqrt(1.0 - 330.0/1000.0)
        old_hs = old_eta * delta_f
        print(f"  Old eta_s (wrong): {old_eta:.4f}")
        print(f"  Old median hs: {np.nanmedian(old_hs[old_hs>0]):.4f} m")
        print(f"  New median hs: {np.nanmedian(hs[valid]):.4f} m")
        print(f"  Reduction: {100*(1 - np.nanmedian(hs[valid])/np.nanmedian(old_hs[old_hs>0])):.1f}%")
else:
    print("No data for this date")

Matched CS2-IS2 pairs: 0


In [5]:
print()
print("=" * 50)
print("WHAT TO EXPECT")
print("=" * 50)
print()
print("If the corrected formula is working:")
print("  1. eta_s should be 1.2634 (NOT 1.222)")
print("  2. Every hs value should be LESS than delta_f")
print("  3. Ratio hs/delta_f should be ~0.792")
print("  4. Snow thickness should be ~35% lower than old values")
print("  5. Median hs for Aug 2022 full month should be ~0.10 m")
print()
print("Paste the output of ALL cells to Claude for verification.")


WHAT TO EXPECT

If the corrected formula is working:
  1. eta_s should be 1.2634 (NOT 1.222)
  2. Every hs value should be LESS than delta_f
  3. Ratio hs/delta_f should be ~0.792
  4. Snow thickness should be ~35% lower than old values
  5. Median hs for Aug 2022 full month should be ~0.10 m

Paste the output of ALL cells to Claude for verification.


In [6]:
# If loaded from batch output:
if 'df_aug' not in dir() or len(df_aug) == 0:
    df_aug = pd.read_csv('/g/data/gv90/xl1657/phd/M2_workspace/output/collocated/collocated_202208.csv')

# Compute snow thickness with corrected formula
hs, hs_unc = compute_snow_thickness(
    df_aug['is2_fb_wm'].values,
    df_aug['cs2_rfb'].values,
    is2_unc=df_aug['is2_fb_unc_mean'].values,
    cs2_unc=0.03 * np.ones(len(df_aug)),
    rho_s=330.0,
    rho_s_unc=70.0,
)

delta_f = df_aug['is2_fb_wm'].values - df_aug['cs2_rfb'].values
valid = hs > 0

print("=" * 60)
print("AUGUST 2022 — CORRECTED FORMULA VERIFICATION")
print("=" * 60)
print(f"Total matchups:  {len(df_aug)}")
print(f"Valid (hs > 0):  {valid.sum()} ({100*valid.sum()/len(hs):.1f}%)")
print(f"Negative:        {(~valid).sum()} ({100*(~valid).sum()/len(hs):.1f}%)")
print()
print(f"Freeboard difference (delta_f):")
print(f"  Median: {np.nanmedian(delta_f):.4f} m")
print(f"  Mean:   {np.nanmean(delta_f):.4f} m")
print()
print(f"CORRECTED snow thickness:")
print(f"  Median (valid): {np.nanmedian(hs[valid]):.4f} m")
print(f"  Mean (valid):   {np.nanmean(hs[valid]):.4f} m")
print()

# Formula ratio check
ratio = np.nanmedian(hs[valid]) / np.nanmedian(delta_f[valid])
print(f"RATIO CHECK:")
print(f"  hs_median / delta_f_median = {ratio:.4f}")
print(f"  Expected (1/1.2628):        {1/1.2628:.4f}")
print(f"  PASS: {abs(ratio - 1/1.2628) < 0.02}")
print()

# Comparison with old wrong values
old_eta = 1.0 / np.sqrt(1.0 - 330.0 / 1000.0)
old_hs = old_eta * delta_f
old_valid = old_hs > 0
print(f"COMPARISON — OLD vs NEW:")
print(f"  Old eta_s (WRONG): {old_eta:.4f} (multiplied delta_f)")
print(f"  New eta_s (CORRECT): 1.2628 (divided delta_f)")
print(f"  Old median hs:  {np.nanmedian(old_hs[old_valid]):.4f} m")
print(f"  New median hs:  {np.nanmedian(hs[valid]):.4f} m")
print(f"  Change: {100*(np.nanmedian(hs[valid])/np.nanmedian(old_hs[old_valid]) - 1):.1f}%")
print()

# Uncertainty
if hs_unc is not None:
    print(f"Uncertainty (3-component):")
    print(f"  Median: {np.nanmedian(hs_unc[valid]):.4f} m")
    print(f"  Mean:   {np.nanmean(hs_unc[valid]):.4f} m")
    print()

# Sector breakdown
def assign_sector(lon):
    if -62 <= lon <= 15:
        return 'Weddell'
    elif (160 <= lon <= 180) or (-180 <= lon <= -140):
        return 'Ross'
    return 'Other'

df_aug['sector'] = df_aug['lon'].apply(assign_sector)
for sector in ['Weddell', 'Ross', 'Other']:
    mask = df_aug['sector'] == sector
    n = mask.sum()
    if n > 0:
        s_hs = hs[mask.values]
        s_valid = s_hs > 0
        print(f"{sector}: n={n}, valid={s_valid.sum()}, "
              f"median hs={np.nanmedian(s_hs[s_valid]):.4f} m")

print()
print("=" * 60)
print("Paste this entire output to Claude for verification")
print("=" * 60)

AUGUST 2022 — CORRECTED FORMULA VERIFICATION
Total matchups:  11822
Valid (hs > 0):  10160 (85.9%)
Negative:        1662 (14.1%)

Freeboard difference (delta_f):
  Median: 0.1257 m
  Mean:   0.1688 m

CORRECTED snow thickness:
  Median (valid): 0.1223 m
  Mean (valid):   0.1683 m

RATIO CHECK:
  hs_median / delta_f_median = 0.7919
  Expected (1/1.2628):        0.7919
  PASS: True

COMPARISON — OLD vs NEW:
  Old eta_s (WRONG): 1.2217 (multiplied delta_f)
  New eta_s (CORRECT): 1.2628 (divided delta_f)
  Old median hs:  0.1886 m
  New median hs:  0.1223 m
  Change: -35.2%

Uncertainty (3-component):
  Median: 0.0269 m
  Mean:   0.0282 m

Weddell: n=5541, valid=4802, median hs=0.1379 m
Ross: n=3391, valid=2926, median hs=0.0991 m
Other: n=2890, valid=2432, median hs=0.1313 m

Paste this entire output to Claude for verification
